# SARSA: On-Policy TD Control

## Problem Statement

I want an agent to learn a good policy in a **4×4 GridWorld** environment
using the **SARSA** algorithm (State–Action–Reward–State–Action), which is
a form of **on-policy Temporal-Difference (TD) control**.

### Environment
- The world is a 4×4 grid of cells. The agent starts at the top-left cell `(0, 0)`.
- The agent can move in 4 directions: **right, left, down, up**.
- Moving off the grid is not allowed (the agent stays in place).
- The **goal** is the bottom-right cell `(3, 3)`.
- Every step costs a reward of **−1** (encourages short paths).
- Reaching the goal gives a reward of **+10** and ends the episode.

### Goal
I want to learn a Q-table `Q(s, a)` such that the greedy policy
`π(s) = argmax_a Q(s, a)` takes the agent from start to goal efficiently.

### Why SARSA (on-policy)?
SARSA updates Q using the **action actually taken by the current
ε-greedy policy**, not the max over actions. This makes it *on-policy*:
it evaluates the policy it is currently following, exploration included.

Update rule:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma\, Q(s', a') - Q(s,a) \big]$$

where `a'` is the action chosen by the current ε-greedy policy at `s'`.

## Step 1 — Define the GridWorld environment

I encode the environment as two small helper functions:
- `step(state, action)` returns `(next_state, reward, done)`.
- `ACTIONS` maps action index → (row, col) delta.`

In [1]:
import numpy as np

GRID = 4
# action 0: right, 1: left, 2: down, 3: up
ACTIONS = [(0, 1), (0, -1), (1, 0), (-1, 0)]

def step(state, action):
    """Take one action in the GridWorld. Return (next_state, reward, done)."""
    r, c = state
    dr, dc = ACTIONS[action]
    nr = max(0, min(GRID - 1, r + dr))
    nc = max(0, min(GRID - 1, c + dc))
    new_state = (nr, nc)
    if new_state == (GRID - 1, GRID - 1):   # reached goal
        return new_state, 10.0, True
    return new_state, -1.0, False

## Step 2 — Set SARSA hyperparameters and initialize the Q-table

| Symbol | Meaning | Value |
|--------|---------|-------|
| α (alpha) | learning rate | 0.1 |
| γ (gamma) | discount factor | 0.9 |
| ε (epsilon) | exploration rate | 0.1 |
| episodes | training episodes | 500 |

I store `Q` as a `4 × 4 × 4` array: one value per (row, col, action).

In [3]:
alpha       = 0.1
gamma       = 0.9
epsilon     = 0.1
n_episodes  = 500

Q = np.zeros((GRID, GRID, len(ACTIONS)))
print("Q shape:", Q.shape)

Q shape: (4, 4, 4)


## Step 3 — ε-greedy action selection

With probability **ε**, I pick a random action (exploration).
Otherwise, I pick the action with the highest Q-value at the current state
(exploitation).

SARSA calls this function **twice per step**: once for `a`, once for `a'`.

In [4]:
def choose_action(state):
    """ε-greedy action selection."""
    if np.random.rand() < epsilon:
        return np.random.randint(len(ACTIONS))     # explore
    return int(np.argmax(Q[state[0], state[1]]))   # exploit

## Step 4 — SARSA training loop

For each episode:

1. I start at `S = (0, 0)` and pick `A` with ε-greedy.
2. I repeat until terminal:
   - I take action `A` → observe `R`, `S'`.
   - If not terminal, I pick `A'` with ε-greedy **from the same policy**.
   - I compute the TD target:
     - terminal: `target = R`
     - otherwise: `target = R + γ · Q(S', A')`
   - I update: `Q(S, A) ← Q(S, A) + α · (target − Q(S, A))`
   - I set `(S, A) ← (S', A')`.

The tuple of five elements **(S, A, R, S', A')** is exactly where the name
SARSA comes from.

In [5]:
for episode in range(n_episodes):
    state  = (0, 0)
    action = choose_action(state)

    done = False
    while not done:
        next_state, reward, done = step(state, action)

        if done:
            target = reward                                   # no next Q
        else:
            next_action = choose_action(next_state)           # choose A'
            target = reward + gamma * Q[next_state[0], next_state[1], next_action]

        # TD update for (S, A)
        Q[state[0], state[1], action] += alpha * (
            target - Q[state[0], state[1], action]
        )

        # Advance to (S', A')
        state = next_state
        if not done:
            action = next_action

print("Training finished.")

Training finished.


## Step 5 — Inspect learned Q-values

The max Q-value per state should increase as we get closer to the goal
(state `(3,3)` never has Q values because it is terminal).

In [7]:
print("Max Q per state:")
print(np.round(Q.max(axis=2), 2))

Max Q per state:
[[ 0.99  1.59 -0.49 -0.46]
 [ 2.43  3.84  4.94  0.14]
 [ 2.88  5.31  7.5   8.15]
 [-0.29  7.48 10.    0.  ]]


## Step 6 — Test the greedy policy

I follow `argmax_a Q(s, a)` from the start and check whether the agent
reaches the goal. A sensible learned policy should take roughly
6 steps (down, down, down, right, right, right) — the Manhattan distance.

In [8]:
state = (0, 0)
path  = [state]
for _ in range(20):
    a = int(np.argmax(Q[state[0], state[1]]))
    state, _, done = step(state, a)
    path.append(state)
    if done:
        break

print("Greedy path:", path)
print("Steps:", len(path) - 1)

Greedy path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (3, 2), (3, 3)]
Steps: 6


## Optional — Compare with Q-learning

SARSA is **on-policy**: the TD target uses `Q(S', A')` where `A'` was
sampled by the *current* ε-greedy policy.

Q-learning is **off-policy**: it uses `max_a Q(S', a)` instead.

One-line difference in the update:

SARSA: target = R + γ · Q(S', A')
Q-learning: target = R + γ · max_a Q(S', a)


Because SARSA uses the actual (possibly exploratory) next action,
it tends to learn **safer** policies — famously, on the *Cliff-Walking*
environment SARSA takes a longer but safer route while Q-learning
walks along the cliff edge.


In [10]:
#  Q-learning for comparison
Qq = np.zeros((GRID, GRID, len(ACTIONS)))

def choose_action_q(state):
    if np.random.rand() < epsilon:
        return np.random.randint(len(ACTIONS))
    return int(np.argmax(Qq[state[0], state[1]]))

for _ in range(n_episodes):
    s = (0, 0)
    done = False
    while not done:
        a = choose_action_q(s)
        s2, r, done = step(s, a)
        target = r if done else r + gamma * np.max(Qq[s2[0], s2[1]])
        Qq[s[0], s[1], a] += alpha * (target - Qq[s[0], s[1], a])
        s = s2

# Compare final max Q-tables
print("SARSA    max-Q:\n", np.round(Q.max(axis=2), 2))
print("\nQ-learning max-Q:\n", np.round(Qq.max(axis=2), 2))

SARSA    max-Q:
 [[ 0.99  1.59 -0.49 -0.46]
 [ 2.43  3.84  4.94  0.14]
 [ 2.88  5.31  7.5   8.15]
 [-0.29  7.48 10.    0.  ]]

Q-learning max-Q:
 [[ 1.81  3.12  1.05 -0.42]
 [ 2.8   4.58  6.03  0.8 ]
 [ 2.88  6.2   8.    8.15]
 [ 0.48  7.62 10.    0.  ]]
